In [1]:
# Imports
import pandas as pd
import numpy as np

### Project Goal: Why did Manchester City finish 1 point ahead of Liverpool in the Premier League 2021-22 season

In the 21-21 Premier league season Manchester City became the champion 1 point ahead of Liverpool. We will analyze and find out the main reasons for this 1 point gap.

Dataset source: https://www.kaggle.com/datasets/azminetoushikwasi/epl-21-22-matches-players

In [2]:
# Load Datasets

all_match_results_table = pd.read_csv('../data/task04/all_match_results.csv')
all_players_stats_table = pd.read_csv('../data/task04/all_players_stats.csv')
points_table = pd.read_csv('../data/task04/points_table.csv')

In [3]:
all_match_results_table.head(10)

,Date,HomeTeam,Result,AwayTeam
0,13-Aug-2021,Brentford,2:0,Arsenal
1,14-Aug-2021,Burnley,1:2,Brighton and Hove Albion
2,14-Aug-2021,Chelsea,3:0,Crystal Palace
3,14-Aug-2021,Everton,3:1,Southampton
4,14-Aug-2021,Leicester City,1:0,Wolverhampton Wanderers
5,14-Aug-2021,Manchester United,5:1,Leeds United
6,14-Aug-2021,Norwich City,0:3,Liverpool
7,14-Aug-2021,Watford,3:2,Aston Villa
8,15-Aug-2021,Newcastle United,2:4,West Ham United
9,15-Aug-2021,Tottenham Hotspur,1:0,Manchester City


In [4]:
all_players_stats_table.head(10)

,Team,JerseyNo,Player,Position,Apearances,Substitutions,Goals,Penalties,YellowCards,RedCards
0,Arsenal,7,Bukayo Saka,Defender/Midfielder,40,3,12,2,6.0,0.0
1,Arsenal,6,Gabriel,Defender,37,1,5,0,7.0,1.0
2,Arsenal,32,Aaron Ramsdale,Goalkeeper,37,0,0,0,1.0,0.0
3,Arsenal,4,Ben White,Defender,37,0,0,0,3.0,0.0
4,Arsenal,8,Martin Odegaard,Midfielder,36,4,7,0,4.0,0.0
5,Arsenal,34,Granit Xhaka,Defender/Midfielder,29,1,1,0,10.0,2.0
6,Arsenal,35,Gabriel Martinelli,Forward,26,10,6,1,2.0,1.0
7,Arsenal,5,Thomas Partey,Midfielder,24,2,2,0,6.0,1.0
8,Arsenal,10,Emile Smith Rowe,Midfielder,24,12,11,0,1.0,0.0
9,Arsenal,3,Kieran Tierney,Defender/Midfielder,24,1,1,0,0.0,0.0


There is a special character('\u00A0') "NBSP" - non-breaking space, which has to be removed

In [5]:
# Removing NBSP
all_players_stats_table['Player'] = (
    all_players_stats_table['Player']
    .str.replace('\u00A0', ' ', regex=False)
    .str.strip()
)

all_players_stats_table.head(10)

,Team,JerseyNo,Player,Position,Apearances,Substitutions,Goals,Penalties,YellowCards,RedCards
0,Arsenal,7,Bukayo Saka,Defender/Midfielder,40,3,12,2,6.0,0.0
1,Arsenal,6,Gabriel,Defender,37,1,5,0,7.0,1.0
2,Arsenal,32,Aaron Ramsdale,Goalkeeper,37,0,0,0,1.0,0.0
3,Arsenal,4,Ben White,Defender,37,0,0,0,3.0,0.0
4,Arsenal,8,Martin Odegaard,Midfielder,36,4,7,0,4.0,0.0
5,Arsenal,34,Granit Xhaka,Defender/Midfielder,29,1,1,0,10.0,2.0
6,Arsenal,35,Gabriel Martinelli,Forward,26,10,6,1,2.0,1.0
7,Arsenal,5,Thomas Partey,Midfielder,24,2,2,0,6.0,1.0
8,Arsenal,10,Emile Smith Rowe,Midfielder,24,12,11,0,1.0,0.0
9,Arsenal,3,Kieran Tierney,Defender/Midfielder,24,1,1,0,0.0,0.0


In [6]:
points_table.head(10)

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts
0,1,Manchester City,38,29,6,3,99,26,73,93
1,2,Liverpool,38,28,8,2,94,26,68,92
2,3,Chelsea,38,21,11,6,76,33,43,74
3,4,Tottenham Hotspur,38,22,5,11,69,40,29,71
4,5,Arsenal,38,22,3,13,61,48,13,69
5,6,Manchester United,38,16,10,12,57,57,0,58
6,7,West Ham United,38,16,8,14,60,51,9,56
7,8,Leicester City,38,14,10,14,62,59,3,52
8,9,Brighton and Hove Albion,38,12,15,11,42,44,-2,51
9,10,Wolverhampton Wanderers,38,15,6,17,38,43,-5,51


In [7]:
# Column inspection
print(all_players_stats_table.columns)
print(all_players_stats_table.columns)
print(points_table.columns)

Index(['Team', 'JerseyNo', 'Player', 'Position', 'Apearances', 'Substitutions',
       'Goals', 'Penalties', 'YellowCards', 'RedCards'],
      dtype='str')
Index(['Team', 'JerseyNo', 'Player', 'Position', 'Apearances', 'Substitutions',
       'Goals', 'Penalties', 'YellowCards', 'RedCards'],
      dtype='str')
Index(['Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts'], dtype='str')


We will rename the columns in more Pythonic way


In [8]:
# Functions
def headers_to_snake_case(df):
    """Renames headers to snake_case"""
    df = df.copy()

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", regex=True)
        .str.replace(r"([a-z0-9])([A-Z])", r"\1_\2", regex=True)
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    return df

def sum_pts(df):
    """Sum points"""
    return df['points'].sum()

def pts_per_game(df, pts):
    """Gets points per game"""
    return pts / len(df)

def home_away_matches(venue, team):
    """Gives the home or away matches of a team"""
    return (
    team_match_results.loc[
    (team_match_results['team'].isin([team])) &
    (team_match_results['venue'] == venue)
]
)


In [9]:
# Renaming columns
all_players_stats_table = headers_to_snake_case(all_players_stats_table)
all_match_results_table = headers_to_snake_case(all_match_results_table)
points_table = headers_to_snake_case(points_table)

for df in [all_players_stats_table, all_match_results_table, points_table]:
    print(df.columns)



Index(['team', 'jersey_no', 'player', 'position', 'apearances',
       'substitutions', 'goals', 'penalties', 'yellow_cards', 'red_cards'],
      dtype='str')
Index(['date', 'home_team', 'result', 'away_team'], dtype='str')
Index(['pos', 'team', 'pld', 'w', 'd', 'l', 'gf', 'ga', 'gd', 'pts'], dtype='str')


In [10]:
print(f"All players stats table shape:{all_players_stats_table.shape}")
print(f"All match results table shape:{all_match_results_table.shape}")
print(f"Points table shape:{points_table.shape}")

All players stats table shape:(623, 10)
All match results table shape:(380, 4)
Points table shape:(20, 10)


In [11]:
all_players_stats_table.head(10)

,team,jersey_no,player,position,apearances,substitutions,goals,penalties,yellow_cards,red_cards
0,Arsenal,7,Bukayo Saka,Defender/Midfielder,40,3,12,2,6.0,0.0
1,Arsenal,6,Gabriel,Defender,37,1,5,0,7.0,1.0
2,Arsenal,32,Aaron Ramsdale,Goalkeeper,37,0,0,0,1.0,0.0
3,Arsenal,4,Ben White,Defender,37,0,0,0,3.0,0.0
4,Arsenal,8,Martin Odegaard,Midfielder,36,4,7,0,4.0,0.0
5,Arsenal,34,Granit Xhaka,Defender/Midfielder,29,1,1,0,10.0,2.0
6,Arsenal,35,Gabriel Martinelli,Forward,26,10,6,1,2.0,1.0
7,Arsenal,5,Thomas Partey,Midfielder,24,2,2,0,6.0,1.0
8,Arsenal,10,Emile Smith Rowe,Midfielder,24,12,11,0,1.0,0.0
9,Arsenal,3,Kieran Tierney,Defender/Midfielder,24,1,1,0,0.0,0.0


#### All Players Stats Table

Columns: Team , Jersey Number, Player Name, Position, Appearances , Substitutions, Goals, Penalties, Yellow Cards, Red Cards
Shape: 623 Rows and 10 Columns

- Team (team): The name of the Team. It is normal to have values with duplicate rows
- Jersey Number (jersey_no): The jersey number of the player. Duplicate values should be checked per team , as in the different teams , there are players with the same jersey numbers.
- Position (position): The column can contain more than one positions , which are divided by "/"
- Appearances (apearances): These are the total appearances in the season for this player, not only in the Premier League. The column will be renamed to "appearances" , as there is a typo
- Substitutions (substitutions): How many times the player was substituted during the season
- Goals (goals): Scored goals by the player
- Penalties (penalties) : These are the scored penalties by the player, not given away penalties.
- Yellow Cards (yellow_cards): Count of the yellow cards received by the player.
- Red Cards (red_cards): Count of the red cards received by the player.

#### Player statistics limitation

The player statistics table was excluded from the analysis because it contains data from multiple competitions and does not include a field that identifies the competition in which each goal or appearance occurred.

As a result, the player statistics cannot be filtered reliably to Premier League matches only. Using these values would risk comparing all-competition player totals with Premier League team totals, which would make the analysis inconsistent and potentially misleading.

Therefore, the final analysis is based only on the league-specific points table and match-results data.

In [12]:
all_players_stats_table = all_players_stats_table.rename(columns={
    'apearances': 'appearances',
})
all_players_stats_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   team           623 non-null    str    
 1   jersey_no      623 non-null    int64  
 2   player         623 non-null    str    
 3   position       623 non-null    str    
 4   appearances    623 non-null    int64  
 5   substitutions  623 non-null    int64  
 6   goals          623 non-null    int64  
 7   penalties      623 non-null    int64  
 8   yellow_cards   623 non-null    float64
 9   red_cards      623 non-null    float64
dtypes: float64(2), int64(5), str(3)
memory usage: 48.8 KB


There is no need that the red and yellow cards to be float. They should be integers, as they are given one by one. There can't be 1.5 card.

In [13]:
cards_cols = ['yellow_cards', 'red_cards']

all_players_stats_table[cards_cols] = (
    all_players_stats_table[cards_cols]
    .apply(pd.to_numeric, errors='coerce')
    .astype('int64')
)

all_players_stats_table[cards_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   yellow_cards  623 non-null    int64
 1   red_cards     623 non-null    int64
dtypes: int64(2)
memory usage: 9.9 KB


There should be 20 unique values in the "team" column, as the Premier League has 20 teams

In [14]:
# Unique Teams
print(f"Unique teams values: {len(all_players_stats_table['team'].unique())}")

Unique teams values: 20


In [15]:
# Missing Values
all_players_stats_table.isna().sum()

team             0
jersey_no        0
player           0
position         0
appearances      0
substitutions    0
goals            0
penalties        0
yellow_cards     0
red_cards        0
dtype: int64

In [16]:
all_match_results_table.head(10)

,date,home_team,result,away_team
0,13-Aug-2021,Brentford,2:0,Arsenal
1,14-Aug-2021,Burnley,1:2,Brighton and Hove Albion
2,14-Aug-2021,Chelsea,3:0,Crystal Palace
3,14-Aug-2021,Everton,3:1,Southampton
4,14-Aug-2021,Leicester City,1:0,Wolverhampton Wanderers
5,14-Aug-2021,Manchester United,5:1,Leeds United
6,14-Aug-2021,Norwich City,0:3,Liverpool
7,14-Aug-2021,Watford,3:2,Aston Villa
8,15-Aug-2021,Newcastle United,2:4,West Ham United
9,15-Aug-2021,Tottenham Hotspur,1:0,Manchester City


In [17]:
all_match_results_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   date       380 non-null    str  
 1   home_team  380 non-null    str  
 2   result     380 non-null    str  
 3   away_team  380 non-null    str  
dtypes: str(4)
memory usage: 12.0 KB


#### All Match Results table
Columns: Date, Home Team, Result, Away Team

- Date (date): This is the match date. The Date column needs to be transformed into a date format
- Home Team (home_team): Name of the home team
- Result (result): Result of the game. The Result column will be divided in two - home score and away score. Both will be int64 fields
- Away Team (away_team): Name of the away team

In [18]:
# Check for null values
all_match_results_table.isna().sum()

date         0
home_team    0
result       0
away_team    0
dtype: int64

In [19]:
# Date column to datetime format
all_match_results_table.date = (
    all_match_results_table.date
    .apply(pd.to_datetime, errors='coerce')
)

all_match_results_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       380 non-null    datetime64[us]
 1   home_team  380 non-null    str           
 2   result     380 non-null    str           
 3   away_team  380 non-null    str           
dtypes: datetime64[us](1), str(3)
memory usage: 12.0 KB


There should be 20 unique values in home_team and away_team, as the Premier League consists of 20 teams

In [20]:
# Unique values
print(f"Home team unique values: {len(all_match_results_table['home_team'].unique())}")
print(f"Away team unique values: {len(all_match_results_table['away_team'].unique())}")


Home team unique values: 20
Away team unique values: 20


We create a new table, which will be modified for our needs

In [21]:
match_results_cleand = all_match_results_table.copy()

In [22]:
# Create a "match_id"  column
match_results_cleand['match_id'] = range(1, len(match_results_cleand) + 1)

In [23]:
# Split result and create new columns
score_split = match_results_cleand.result.str.split(':', expand=True)

match_results_cleand['home_goals'] = score_split[0].astype(int)
match_results_cleand['away_goals'] = score_split[1].astype(int)

match_results_cleand

,date,home_team,result,away_team,match_id,home_goals,away_goals
0,2021-08-13,Brentford,2:0,Arsenal,1,2,0
1,2021-08-14,Burnley,1:2,Brighton and Hove Albion,2,1,2
2,2021-08-14,Chelsea,3:0,Crystal Palace,3,3,0
3,2021-08-14,Everton,3:1,Southampton,4,3,1
4,2021-08-14,Leicester City,1:0,Wolverhampton Wanderers,5,1,0
...,...,...,...,...,...,...,...
375,2022-05-22,Crystal Palace,1:0,Manchester United,376,1,0
376,2022-05-22,Leicester City,4:1,Southampton,377,4,1
377,2022-05-22,Liverpool,3:1,Wolverhampton Wanderers,378,3,1
378,2022-05-22,Manchester City,3:2,Aston Villa,379,3,2


In [24]:
# Add results columns
results_mapper = {
    1: 'Win',
    0: 'Draw',
    -1: 'Loss',
}

points_mapper = {
   "Win": 3,
    "Draw": 1,
    "Loss": 0,
}

match_results_cleand['home_result'] = np.sign(
    match_results_cleand['home_goals'] - match_results_cleand['away_goals']
).map(results_mapper)

match_results_cleand['away_result'] = np.sign(
    match_results_cleand['away_goals'] - match_results_cleand['home_goals']
).map(results_mapper)

match_results_cleand['home_points'] = match_results_cleand['home_result'].map(points_mapper)
match_results_cleand['away_points'] = match_results_cleand['away_result'].map(points_mapper)

match_results_cleand.head(100)


,date,home_team,result,away_team,match_id,home_goals,away_goals,home_result,away_result,home_points,away_points
0,2021-08-13,Brentford,2:0,Arsenal,1,2,0,Win,Loss,3,0
1,2021-08-14,Burnley,1:2,Brighton and Hove Albion,2,1,2,Loss,Win,0,3
2,2021-08-14,Chelsea,3:0,Crystal Palace,3,3,0,Win,Loss,3,0
3,2021-08-14,Everton,3:1,Southampton,4,3,1,Win,Loss,3,0
4,2021-08-14,Leicester City,1:0,Wolverhampton Wanderers,5,1,0,Win,Loss,3,0
...,...,...,...,...,...,...,...,...,...,...,...
95,2021-10-30,Tottenham Hotspur,0:3,Manchester United,96,0,3,Loss,Win,0,3
96,2021-10-30,Watford,0:1,Southampton,97,0,1,Loss,Win,0,3
97,2021-10-31,Aston Villa,1:4,West Ham United,98,1,4,Loss,Win,0,3
98,2021-10-31,Norwich City,1:2,Leeds United,99,1,2,Loss,Win,0,3


In [25]:
points_table

,pos,team,pld,w,d,l,gf,ga,gd,pts
0,1,Manchester City,38,29,6,3,99,26,73,93
1,2,Liverpool,38,28,8,2,94,26,68,92
2,3,Chelsea,38,21,11,6,76,33,43,74
3,4,Tottenham Hotspur,38,22,5,11,69,40,29,71
4,5,Arsenal,38,22,3,13,61,48,13,69
5,6,Manchester United,38,16,10,12,57,57,0,58
6,7,West Ham United,38,16,8,14,60,51,9,56
7,8,Leicester City,38,14,10,14,62,59,3,52
8,9,Brighton and Hove Albion,38,12,15,11,42,44,-2,51
9,10,Wolverhampton Wanderers,38,15,6,17,38,43,-5,51


In [26]:
points_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   pos     20 non-null     int64
 1   team    20 non-null     str  
 2   pld     20 non-null     int64
 3   w       20 non-null     int64
 4   d       20 non-null     int64
 5   l       20 non-null     int64
 6   gf      20 non-null     int64
 7   ga      20 non-null     int64
 8   gd      20 non-null     int64
 9   pts     20 non-null     int64
dtypes: int64(9), str(1)
memory usage: 1.7 KB


#### Points table
Columns: Position (pos), Team (team), Played (pld), Wins (w), Draws (d), Looses (l), Goals For (gf), Goals against (ga), Goal difference (gd), Points (pts)

- Position (pos) - Shows the position on which the team has ended the season
- Team (team) = The name of the team
- Played (pld) -  Games played in the season (should be 38)
- Wins (w) - Wins count
- Draws (d) - Draws count
- Looses (l) - Looses count
- Goals For (gf) - Count of goals scored against other teams
- Goals against (ga) - Conceded goals
- Goal difference (gd) - Goal differences
- Points (pts) - Points gained during the season ( Win: 3, Draw: 1, Loose: 0)

In [27]:
# Check for null values
points_table.isna().sum()

pos     0
team    0
pld     0
w       0
d       0
l       0
gf      0
ga      0
gd      0
pts     0
dtype: int64

No null values. Only the team column is str, all others are int64

In [28]:
print(f'Teams (should be 20): {len(points_table['team'].unique())}')
print(f'Teams with 38 games (should be 20): {len(points_table.loc[points_table['pld'] == 38, 'team'])}')

Teams (should be 20): 20
Teams with 38 games (should be 20): 20


In [29]:
# Compare match table cleaned and the points table
liverpool_points = match_results_cleand.loc[
    match_results_cleand['home_team'] == 'Liverpool',
    'home_points'
].sum() + match_results_cleand.loc[
    match_results_cleand['away_team'] == 'Liverpool',
    'away_points'].sum()

man_city_points = match_results_cleand.loc[
    match_results_cleand['home_team'] == 'Manchester City',
    'home_points'
].sum() + match_results_cleand.loc[
    match_results_cleand['away_team'] == 'Manchester City',
    'away_points'].sum()

liv_man_points_table = points_table.loc[
    points_table['team'].isin(['Liverpool', 'Manchester City']),
]

print(f'Liverpool points in cleand table: {liverpool_points}')
print(f'Manchester city points in cleand table: {man_city_points}')

print(liv_man_points_table)



Liverpool points in cleand table: 92
Manchester city points in cleand table: 93
   pos             team  pld   w  d  l  gf  ga  gd  pts
0    1  Manchester City   38  29  6  3  99  26  73   93
1    2        Liverpool   38  28  8  2  94  26  68   92


Data is valid as in both tables Liverpool has 92 points and Manchester City has 93 points.

In [30]:
liv_man_points_table


,pos,team,pld,w,d,l,gf,ga,gd,pts
0,1,Manchester City,38,29,6,3,99,26,73,93
1,2,Liverpool,38,28,8,2,94,26,68,92


#### Final table comparison
The final league table shows that Manchester City and Liverpool had very similar overall performances during the 2021–22 season. Both teams played 38 matches and conceded 26 goals, which suggests that their defensive records were equally strong.

Liverpool lost one fewer match than Manchester City, with 2 losses compared with City's 3. However, Manchester City won one more match and recorded two fewer draws. City finished with 29 wins and 6 draws, while Liverpool recorded 28 wins and 8 draws. This indicates that Manchester City converted slightly more matches into wins, whereas Liverpool drew more often.

Manchester City also scored 99 goals, five more than Liverpool's 94, and finished with a goal difference of +73 compared with Liverpool's +68. Based on the available data, City's slightly stronger attacking output and their ability to convert more matches into wins are likely measurable explanations for their one-point advantage at the end of the season.

However, the final standings alone do not show which specific matches created the difference. Further analysis of dropped points and match-level results is needed to explain where the one-point gap occurred.

In [31]:
# Liverpool dropped points table

liv_dropped_points = match_results_cleand.loc[
    ((match_results_cleand['home_team'] == 'Liverpool')
    & (match_results_cleand['home_result'] != 'Win'))
    |
    ((match_results_cleand['away_team'] == 'Liverpool')
    & (match_results_cleand['away_result'] != 'Win')),
]

liv_dropped_points

,date,home_team,result,away_team,match_id,home_goals,away_goals,home_result,away_result,home_points,away_points
22,2021-08-28,Liverpool,1:1,Chelsea,23,1,1,Draw,Draw,1,1
50,2021-09-25,Brentford,3:3,Liverpool,51,3,3,Draw,Draw,1,1
67,2021-10-03,Liverpool,2:2,Manchester City,68,2,2,Draw,Draw,1,1
92,2021-10-30,Liverpool,2:2,Brighton and Hove Albion,93,2,2,Draw,Draw,1,1
109,2021-11-07,West Ham United,3:2,Liverpool,110,3,2,Win,Loss,3,0
167,2021-12-19,Tottenham Hotspur,2:2,Liverpool,168,2,2,Draw,Draw,1,1
177,2021-12-28,Leicester City,1:0,Liverpool,178,1,0,Win,Loss,3,0
187,2022-01-02,Chelsea,2:2,Liverpool,188,2,2,Draw,Draw,1,1
307,2022-04-10,Manchester City,2:2,Liverpool,308,2,2,Draw,Draw,1,1
347,2022-05-07,Liverpool,1:1,Tottenham Hotspur,348,1,1,Draw,Draw,1,1


In [32]:
print(match_results_cleand.head(10))

        date          home_team result                 away_team  match_id  \
0 2021-08-13          Brentford    2:0                   Arsenal         1   
1 2021-08-14            Burnley    1:2  Brighton and Hove Albion         2   
2 2021-08-14            Chelsea    3:0            Crystal Palace         3   
3 2021-08-14            Everton    3:1               Southampton         4   
4 2021-08-14     Leicester City    1:0   Wolverhampton Wanderers         5   
5 2021-08-14  Manchester United    5:1              Leeds United         6   
6 2021-08-14       Norwich City    0:3                 Liverpool         7   
7 2021-08-14            Watford    3:2               Aston Villa         8   
8 2021-08-15   Newcastle United    2:4           West Ham United         9   
9 2021-08-15  Tottenham Hotspur    1:0           Manchester City        10   

   home_goals  away_goals home_result away_result  home_points  away_points  
0           2           0         Win        Loss            3 

In [33]:
home_rows = match_results_cleand[
    [
        "match_id",
        "date",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "home_result",
        "home_points"
    ]
]

home_rows = home_rows.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_goals": "goals_for",
        "away_goals": "goals_against",
        "home_result": "result",
        "home_points": "points"
    }
)

home_rows["venue"] = "Home"

away_rows = match_results_cleand[
    [
        "match_id",
        "date",
        "away_team",
        "home_team",
        "away_goals",
        "home_goals",
        "away_result",
        "away_points"
    ]
]

away_rows = away_rows.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_goals": "goals_for",
        "home_goals": "goals_against",
        "away_result": "result",
        "away_points": "points"
    }
)

away_rows["venue"] = "Away"

team_match_results = pd.concat(
    [home_rows, away_rows],
    ignore_index=True,
)

team_match_results

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue
0,1,2021-08-13,Brentford,Arsenal,2,0,Win,3,Home
1,2,2021-08-14,Burnley,Brighton and Hove Albion,1,2,Loss,0,Home
2,3,2021-08-14,Chelsea,Crystal Palace,3,0,Win,3,Home
3,4,2021-08-14,Everton,Southampton,3,1,Win,3,Home
4,5,2021-08-14,Leicester City,Wolverhampton Wanderers,1,0,Win,3,Home
...,...,...,...,...,...,...,...,...,...
755,376,2022-05-22,Manchester United,Crystal Palace,0,1,Loss,0,Away
756,377,2022-05-22,Southampton,Leicester City,1,4,Loss,0,Away
757,378,2022-05-22,Wolverhampton Wanderers,Liverpool,1,3,Loss,0,Away
758,379,2022-05-22,Aston Villa,Manchester City,2,3,Loss,0,Away


In [34]:
team_match_results['goal_difference'] = team_match_results['goals_for'] - team_match_results['goals_against']
team_match_results

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue,goal_difference
0,1,2021-08-13,Brentford,Arsenal,2,0,Win,3,Home,2
1,2,2021-08-14,Burnley,Brighton and Hove Albion,1,2,Loss,0,Home,-1
2,3,2021-08-14,Chelsea,Crystal Palace,3,0,Win,3,Home,3
3,4,2021-08-14,Everton,Southampton,3,1,Win,3,Home,2
4,5,2021-08-14,Leicester City,Wolverhampton Wanderers,1,0,Win,3,Home,1
...,...,...,...,...,...,...,...,...,...,...
755,376,2022-05-22,Manchester United,Crystal Palace,0,1,Loss,0,Away,-1
756,377,2022-05-22,Southampton,Leicester City,1,4,Loss,0,Away,-3
757,378,2022-05-22,Wolverhampton Wanderers,Liverpool,1,3,Loss,0,Away,-2
758,379,2022-05-22,Aston Villa,Manchester City,2,3,Loss,0,Away,-1


In [35]:
team_match_results['dropped_points'] = 3 - team_match_results['points']
team_match_results

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue,goal_difference,dropped_points
0,1,2021-08-13,Brentford,Arsenal,2,0,Win,3,Home,2,0
1,2,2021-08-14,Burnley,Brighton and Hove Albion,1,2,Loss,0,Home,-1,3
2,3,2021-08-14,Chelsea,Crystal Palace,3,0,Win,3,Home,3,0
3,4,2021-08-14,Everton,Southampton,3,1,Win,3,Home,2,0
4,5,2021-08-14,Leicester City,Wolverhampton Wanderers,1,0,Win,3,Home,1,0
...,...,...,...,...,...,...,...,...,...,...,...
755,376,2022-05-22,Manchester United,Crystal Palace,0,1,Loss,0,Away,-1,3
756,377,2022-05-22,Southampton,Leicester City,1,4,Loss,0,Away,-3,3
757,378,2022-05-22,Wolverhampton Wanderers,Liverpool,1,3,Loss,0,Away,-2,3
758,379,2022-05-22,Aston Villa,Manchester City,2,3,Loss,0,Away,-1,3


In [36]:
team_match_results['clean_sheet'] = team_match_results['goals_against'] == 0
team_match_results['failed_to_score'] = team_match_results['goals_for'] == 0
team_match_results['one_goal_game'] = (team_match_results['goals_for'] - team_match_results['goals_against']).abs() == 1


team_match_results

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue,goal_difference,dropped_points,clean_sheet,failed_to_score,one_goal_game
0,1,2021-08-13,Brentford,Arsenal,2,0,Win,3,Home,2,0,True,False,False
1,2,2021-08-14,Burnley,Brighton and Hove Albion,1,2,Loss,0,Home,-1,3,False,False,True
2,3,2021-08-14,Chelsea,Crystal Palace,3,0,Win,3,Home,3,0,True,False,False
3,4,2021-08-14,Everton,Southampton,3,1,Win,3,Home,2,0,False,False,False
4,5,2021-08-14,Leicester City,Wolverhampton Wanderers,1,0,Win,3,Home,1,0,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,376,2022-05-22,Manchester United,Crystal Palace,0,1,Loss,0,Away,-1,3,False,True,True
756,377,2022-05-22,Southampton,Leicester City,1,4,Loss,0,Away,-3,3,False,False,False
757,378,2022-05-22,Wolverhampton Wanderers,Liverpool,1,3,Loss,0,Away,-2,3,False,False,False
758,379,2022-05-22,Aston Villa,Manchester City,2,3,Loss,0,Away,-1,3,False,False,True


In [37]:
team_stats = (
    team_match_results
    .groupby('team')
    .agg(
        dropped_points_total=('dropped_points', 'sum'),
        clean_sheets_total=('clean_sheet', 'sum'),
        failed_to_score_total=('failed_to_score', 'sum'),
    )
    .reset_index()
).sort_values(by='dropped_points_total')

liv_city_stats = team_stats.loc[team_stats['team'].isin(['Liverpool', 'Manchester City'])]

liv_city_stats

,team,dropped_points_total,clean_sheets_total,failed_to_score_total
11,Manchester City,21,21,4
10,Liverpool,22,21,1


In [38]:
man_city_dropped_points = team_match_results.loc[(team_match_results['team'] == 'Manchester City') & (team_match_results['dropped_points'] > 0)]

man_city_dropped_points

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue,goal_difference,dropped_points,clean_sheet,failed_to_score,one_goal_game
44,45,2021-09-18,Manchester City,Southampton,0,0,Draw,1,Home,0,2,True,True,False
93,94,2021-10-30,Manchester City,Crystal Palace,0,2,Loss,0,Home,-2,3,False,True,False
239,240,2022-02-19,Manchester City,Tottenham Hotspur,2,3,Loss,0,Home,-1,3,False,False,True
307,308,2022-04-10,Manchester City,Liverpool,2,2,Draw,1,Home,0,2,False,False,False
389,10,2021-08-15,Manchester City,Tottenham Hotspur,0,1,Loss,0,Away,-1,3,False,True,True
447,68,2021-10-03,Manchester City,Liverpool,2,2,Draw,1,Away,0,2,False,False,False
589,210,2022-01-22,Manchester City,Southampton,1,1,Draw,1,Away,0,2,False,False,False
660,281,2022-03-14,Manchester City,Crystal Palace,0,0,Draw,1,Away,0,2,True,True,False
743,364,2022-05-15,Manchester City,West Ham United,2,2,Draw,1,Away,0,2,False,False,False


In [39]:
liv_dropped_points = team_match_results.loc[(team_match_results['team'] == 'Liverpool') & (team_match_results['dropped_points'] > 0)]

liv_dropped_points

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue,goal_difference,dropped_points,clean_sheet,failed_to_score,one_goal_game
22,23,2021-08-28,Liverpool,Chelsea,1,1,Draw,1,Home,0,2,False,False,False
67,68,2021-10-03,Liverpool,Manchester City,2,2,Draw,1,Home,0,2,False,False,False
92,93,2021-10-30,Liverpool,Brighton and Hove Albion,2,2,Draw,1,Home,0,2,False,False,False
347,348,2022-05-07,Liverpool,Tottenham Hotspur,1,1,Draw,1,Home,0,2,False,False,False
430,51,2021-09-25,Liverpool,Brentford,3,3,Draw,1,Away,0,2,False,False,False
489,110,2021-11-07,Liverpool,West Ham United,2,3,Loss,0,Away,-1,3,False,False,True
547,168,2021-12-19,Liverpool,Tottenham Hotspur,2,2,Draw,1,Away,0,2,False,False,False
557,178,2021-12-28,Liverpool,Leicester City,0,1,Loss,0,Away,-1,3,False,True,True
567,188,2022-01-02,Liverpool,Chelsea,2,2,Draw,1,Away,0,2,False,False,False
687,308,2022-04-10,Liverpool,Manchester City,2,2,Draw,1,Away,0,2,False,False,False


In [40]:
print(f'Man City dropped points in {len(man_city_dropped_points)} games and Liverpool dropped points in {len(liv_dropped_points)} games')

Man City dropped points in 9 games and Liverpool dropped points in 10 games


In [41]:
print(f'Liverpool dropped points against: {', '.join(set(liv_dropped_points['opponent']))}')
print(f'Man City dropped points against: {', '.join(set(man_city_dropped_points['opponent']))}')

Liverpool dropped points against: Leicester City, Brentford, Tottenham Hotspur, Manchester City, Chelsea, Brighton and Hove Albion, West Ham United
Man City dropped points against: Liverpool, Tottenham Hotspur, Crystal Palace, West Ham United, Southampton


In [42]:
liv_losses = liv_dropped_points.loc[liv_dropped_points['result'] == 'Loss']
liv_draws = liv_dropped_points.loc[liv_dropped_points['result'] == 'Draw']

man_city_losses = man_city_dropped_points.loc[man_city_dropped_points['result'] == 'Loss']
man_city_draws = man_city_dropped_points.loc[man_city_dropped_points['result'] == 'Draw']

liv_draws_lost_points = len(liv_draws ) * 2
liv_losses_lost_points = len(liv_losses) * 3

man_city_draws_lost_points = len(man_city_draws) * 2
man_city_losses_lost_points = len(man_city_losses) * 3

print(f'Liverpool - Draws: {len(liv_draws)} => lost points: {liv_draws_lost_points}; Losses: {len(liv_losses)} => '
      f'lost points: {liv_losses_lost_points}')
print(f'Man City - Draws: {len(man_city_draws)} => lost points: {man_city_draws_lost_points}; Losses: {len(man_city_losses)} => lost points: {man_city_losses_lost_points}')

Liverpool - Draws: 8 => lost points: 16; Losses: 2 => lost points: 6
Man City - Draws: 6 => lost points: 12; Losses: 3 => lost points: 9


In [43]:
city_liverpool_dropped_points = (
    team_match_results.loc[
        team_match_results["team"].isin(["Manchester City", "Liverpool"])
    ]
    .groupby(["team", "venue"])["dropped_points"]
    .sum()
    .unstack(fill_value=0)
    .rename(
        columns={
            "Home": "home_dropped_points",
            "Away": "away_dropped_points"
        }
    )
    .reset_index()
)

city_liverpool_dropped_points

venue,team,away_dropped_points,home_dropped_points
0,Liverpool,14,8
1,Manchester City,11,10


In [44]:
city_liverpool_dropped_points['home_dropped_%'] = (
    city_liverpool_dropped_points['home_dropped_points']  /
    (city_liverpool_dropped_points['home_dropped_points'] + city_liverpool_dropped_points['away_dropped_points'])
)

city_liverpool_dropped_points['away_dropped_%'] = (
    city_liverpool_dropped_points['away_dropped_points']  /
    (city_liverpool_dropped_points['home_dropped_points'] + city_liverpool_dropped_points['away_dropped_points'])
)
city_liverpool_dropped_points

venue,team,away_dropped_points,home_dropped_points,home_dropped_%,away_dropped_%
0,Liverpool,14,8,0.363636,0.636364
1,Manchester City,11,10,0.476190,0.523810


#### Dropped points analysis

Manchester City failed to win in 9 matches, while Liverpool failed to win in 10. This means Liverpool dropped points in one additional match over the course of the season.

Manchester City dropped points against five different opponents: Tottenham Hotspur, West Ham United, Liverpool, Crystal Palace, and Southampton.

Liverpool dropped points against seven different opponents: Manchester City, Chelsea, Tottenham Hotspur, West Ham United, Brighton and Hove Albion, Brentford, and Leicester City.

For both teams, most dropped points came from draws rather than defeats:

Manchester City dropped 12 points through draws.
Liverpool dropped 16 points through draws.

This supports the idea that Liverpool’s greater number of draws was an important measurable difference between the two teams. Liverpool recorded eight draws during the season, compared with Manchester City’s six.

Both teams dropped a larger share of their points away from home. Liverpool dropped approximately 64% of their total dropped points in away matches. Manchester City’s dropped points were more evenly distributed, with approximately 47% dropped at home and 53% away.

Based on the available data, Liverpool’s away performance appears to have been a more important weakness. Their larger number of away dropped points, particularly through draws, contributed to the final one-point gap. However, the data does not prove that improving one specific away result would necessarily have changed the title outcome, only that away matches were where Liverpool lost the greatest share of possible points.

In [45]:
# Head-to-head
head_to_head = team_match_results.loc[
    (team_match_results['team'].isin(['Manchester City'])) &
    (team_match_results['opponent'].isin(['Liverpool']))
]

head_to_head

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue,goal_difference,dropped_points,clean_sheet,failed_to_score,one_goal_game
307,308,2022-04-10,Manchester City,Liverpool,2,2,Draw,1,Home,0,2,False,False,False
447,68,2021-10-03,Manchester City,Liverpool,2,2,Draw,1,Away,0,2,False,False,False


In [46]:
h2h_dropped_points = head_to_head['dropped_points'].sum()
print(f'Man City dropped {h2h_dropped_points} against Liverpool from possible 6.')

Man City dropped 4 against Liverpool from possible 6.


#### Head-to-head
Both Premier League matches between Manchester City and Liverpool ended in draws. As a result, each team earned two points from the head-to-head fixtures, so neither side gained an advantage over the other in those matches.

Therefore, the one-point difference in the final standings was not created directly by the head-to-head results. Instead, it came from the teams’ performances against the rest of the league.

In [47]:
liv_home_matches = home_away_matches('Home', 'Liverpool')
liv_away_matches = home_away_matches('Away', 'Liverpool')

man_city_home_matches = home_away_matches('Home', 'Manchester City')
man_city_away_matches = home_away_matches('Away', 'Manchester City')


In [48]:
liv_home_pts = sum_pts(liv_home_matches)
liv_away_pts = sum_pts(liv_away_matches)

man_city_home_pts = sum_pts(man_city_home_matches)
man_city_away_pts = sum_pts(man_city_away_matches)

print(f'Liverpool home points: {liv_home_pts}\nLiverpool away points: {liv_away_pts}')
print('----------------------')
print(f'Man City home points: {man_city_home_pts}\nMan City away points: {man_city_away_pts}')

Liverpool home points: 49
Liverpool away points: 43
----------------------
Man City home points: 47
Man City away points: 46


In [49]:
liv_home_pts_per_game = pts_per_game(liv_home_matches, liv_home_pts)
liv_away_pts_per_game = pts_per_game(liv_away_matches, liv_away_pts)

man_city_home_pts_per_game = pts_per_game(man_city_home_matches, man_city_home_pts)
man_city_away_pts_per_game = pts_per_game(man_city_away_matches, man_city_away_pts)

print(f'Liverpool home points per game: {liv_home_pts_per_game}\nLiverpool away points per game: {liv_away_pts_per_game}')
print('-------------------------')
print(f'Man City home points per game: {man_city_home_pts_per_game}\nMan City away points per game: {man_city_away_pts_per_game}')

Liverpool home points per game: 2.5789473684210527
Liverpool away points per game: 2.263157894736842
-------------------------
Man City home points per game: 2.473684210526316
Man City away points per game: 2.4210526315789473


In [50]:
away_pts_diff = (man_city_away_pts_per_game - liv_away_pts_per_game) * 19
away_pts_diff

np.float64(3.000000000000001)

In [51]:
home_pts_diff = (liv_home_pts_per_game - man_city_home_pts_per_game) * 19
home_pts_diff

np.float64(1.9999999999999978)

#### Home-Away performance
Liverpool performed better at home, earning approximately two more home points than Manchester City. However, Manchester City earned three more points away from home.

As a result, City’s three-point away advantage was enough to offset Liverpool’s two-point home advantage, leaving Manchester City with a net advantage of one point overall.

This means that the final one-point gap can be explained directly through the teams’ home and away point totals. Liverpool were stronger at home, but Manchester City’s superior away performance made the decisive difference.

In [52]:
one_goal_matches = team_match_results.loc[
    (team_match_results['team'].isin(['Liverpool', 'Manchester City']))
        &
    (team_match_results['one_goal_game'])
]

one_goal_matches.loc[one_goal_matches['team'] == 'Manchester City']

,match_id,date,team,opponent,goals_for,goals_against,result,points,venue,goal_difference,dropped_points,clean_sheet,failed_to_score,one_goal_game
128,129,2021-11-28,Manchester City,West Ham United,2,1,Win,3,Home,1,0,False,False,True
153,154,2021-12-11,Manchester City,Wolverhampton Wanderers,1,0,Win,3,Home,1,0,True,False,True
195,196,2022-01-15,Manchester City,Chelsea,1,0,Win,3,Home,1,0,True,False,True
239,240,2022-02-19,Manchester City,Tottenham Hotspur,2,3,Loss,0,Home,-1,3,False,False,True
378,379,2022-05-22,Manchester City,Aston Villa,3,2,Win,3,Home,1,0,False,False,True
389,10,2021-08-15,Manchester City,Tottenham Hotspur,0,1,Loss,0,Away,-1,3,False,True,True
414,35,2021-09-11,Manchester City,Leicester City,1,0,Win,3,Away,1,0,True,False,True
431,52,2021-09-25,Manchester City,Chelsea,1,0,Win,3,Away,1,0,True,False,True
511,132,2021-12-01,Manchester City,Aston Villa,2,1,Win,3,Away,1,0,False,False,True
560,181,2021-12-29,Manchester City,Brentford,1,0,Win,3,Away,1,0,True,False,True


In [53]:
one_goal_summary = (
    one_goal_matches
    .groupby(["team", "result"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

one_goal_summary

result,team,Loss,Win
0,Liverpool,2,7
1,Manchester City,2,10


#### Close matches
Manchester City won 10 matches by a one-goal margin, while Liverpool won 7. Both teams lost 2 matches by a one-goal margin.

This suggests that Manchester City were more successful in narrowly decided matches. City recorded three more one-goal wins, which may indicate a greater ability to convert close games into victories.

However, this does not by itself prove that close matches caused the final one-point gap. It should be considered together with the broader finding that Manchester City won one more league match overall and Liverpool recorded two more draws.

In [54]:
attack_defense_summary = team_match_results.loc[
    team_match_results['team'].isin(['Liverpool', 'Manchester City'])
].groupby('team').agg(
        gf_total=('goals_for', 'sum'),
        ga_total=('goals_against', 'sum'),
        goal_diff_total=('goal_difference', 'sum'),
        clean_sheets_total=('clean_sheet', 'sum'),
        failed_to_score_total=('failed_to_score', 'sum'),
        avg_goals_per_match=('goals_for', 'mean'),
        avg_conceded=('goals_against', 'mean'),
    ).reset_index()

attack_defense_summary

,team,gf_total,ga_total,goal_diff_total,clean_sheets_total,failed_to_score_total,avg_goals_per_match,avg_conceded
0,Liverpool,94,26,68,21,1,2.473684,0.684211
1,Manchester City,99,26,73,21,4,2.605263,0.684211


#### Attach and Defense comparison
Manchester City scored 5 more goals than Liverpool, finishing the season with 99 goals compared with Liverpool’s 94. City also had a higher average goals-per-match rate of 2.61, compared with Liverpool’s 2.47.

However, Liverpool failed to score in only 1 match, while Manchester City failed to score in 4. This suggests that Liverpool scored more consistently across the season, even though City produced the higher total attacking output.

Defensively, the two teams had identical overall records: both conceded 26 goals and kept 21 clean sheets. Therefore, the defensive performance does not appear to explain the one-point gap.

Based on the available data, Manchester City’s slightly stronger overall attacking output can be considered one of the measurable factors that contributed to their title win. However, the extra goals alone did not necessarily create the one-point difference, so this finding should be considered alongside City’s higher number of wins and stronger away performance.

#### Final Conclusion

Based on the available data, the clearest explanation for the one-point gap is that Manchester City dropped one fewer point over the course of the season: 21 compared with Liverpool’s 22.

Manchester City’s stronger away performance appears to have been an important factor. Liverpool earned two more points at home, but City earned three more points away, creating the final net advantage of one point.

City also had a slightly stronger attacking record, scoring 99 goals compared with Liverpool’s 94. However, defensively the teams were equally strong, as both conceded 26 goals and kept 21 clean sheets.

Overall, the available evidence suggests that Manchester City won the title because they were marginally more effective at converting matches into wins, particularly away from home, while also maintaining a small attacking advantage. The data does not prove perfect causality, but these are the strongest measurable reasons for the final one-point difference.